# Battery-aware Colab Runner

배터리 상태(`BatteryShapeFormationEnv`)를 추가한 학습/평가용 노트북.

- 사용 파일: `formation_seq.py`, `comm_env.py`, `comm_train_battery.py`, `comm_eval_battery.py`
- 각 드론에 배터리 잔량 ∈ [0, 1]이 있고 호버시 적게, 이동시 더 많이 줄어듬
- 배터리가 낮은 드론이 움직이면 `low_battery_move_penalty * (1 - battery)`만큼 페널티 → 군집 내 균등 사용 유도
- 평가용 GIF는 매 프레임 각 드론의 배터리 잔량을 색상 막대 + 퍼센트로 표시

런타임: `런타임 → 런타임 유형 변경 → GPU (T4)` 권장

## 1. 기존 폴더 제거 후 GitHub clone

`Saehoon` 브랜치에 배터리 코드가 있음. 다른 브랜치를 쓰는 경우 아래 `BRANCH` 변수만 바꾸면 됨.

In [ ]:
BRANCH = "Saehoon"

!rm -rf RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git
%cd RL-2026s1-tp

!git fetch origin
!git switch $BRANCH
!git pull origin $BRANCH

## 2. Google Drive 저장 경로

체크포인트/GIF/평가 로그를 Drive에 함께 백업해두면 세션이 끊겨도 유지됨.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/drone_results/battery/gifs
!mkdir -p /content/drive/MyDrive/drone_results/battery/evals
!mkdir -p /content/drive/MyDrive/drone_results/battery/ckpts

## 3. 패키지 설치

In [ ]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 4. Smoke test: 배터리 환경 임포트 + 한 스텝

`BatteryShapeFormationEnv`가 정상적으로 로드되고 배터리가 줄어드는지 확인.

In [ ]:
%cd /content/RL-2026s1-tp

import inspect
import comm_env
from comm_env import BatteryShapeFormationEnv, ShapeFormationEnv

print("comm_env file:", comm_env.__file__)
print("BatteryShapeFormationEnv signature:")
print(inspect.signature(BatteryShapeFormationEnv))

env = BatteryShapeFormationEnv(grid_size=25, n_agents=14, max_steps=10)
base = ShapeFormationEnv(grid_size=25, n_agents=14, max_steps=10)
print("baseline obs_dim:", base.obs_dim)
print("battery  obs_dim:", env.obs_dim, "(expected", base.obs_dim + 1 + (14 - 1), ")")

obs, infos = env.reset(seed=0)
print("all start full (1.0):", set(round(env.battery[a], 5) for a in env.possible_agents))

# 절반은 호버(0), 절반은 이동(4)
acts = {a: (0 if i % 2 == 0 else 4) for i, a in enumerate(env.possible_agents)}
obs2, rewards, term, trunc, infos = env.step(acts)
for i, a in enumerate(env.possible_agents):
    print(f"  {a:9s} act={acts[a]}  battery={env.battery[a]:.5f}  reward={rewards[a]:+.4f}")
print("호버 드론 배터리 감소량 ~ hover_battery_cost, 이동 드론 ~ move_battery_cost")

## 5. 빠른 학습 테스트 (sanity check)

한두 분 안에 끝나는 짧은 학습. reward가 음수에서 점차 증가하는지만 확인.

In [ ]:
%cd /content/RL-2026s1-tp
!python comm_train_battery.py \
  --grid-size 25 \
  --n-agents 14 \
  --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.2 \
  --hover-penalty 0.05 \
  --shaping-coef 0.3 \
  --initial-battery 1.0 \
  --hover-battery-cost 0.002 \
  --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.1 \
  --total-frames 8192 \
  --frames-per-batch 1024 \
  --minibatch-size 256 \
  --ppo-epochs 5 \
  --lr 2e-4 \
  --ent-coef 0.015 \
  --ckpt-every 4 \
  --save-dir checkpoints_battery_fast \
  --tb-logdir runs_battery_fast

## 6. 본 학습: GROUND → A (배터리 인지)

배터리 파라미터 가이드:
- `initial-battery`: 시작 배터리 (기본 1.0). 짧은 에피소드면 더 낮춰서 (예: 0.6) 빨리 고갈되게 해 학습 신호를 키울 수 있음.
- `hover-battery-cost` / `move-battery-cost`: 둘 사이 비율이 클수록 "가만히 있기"가 유리해짐. 균등 사용을 강조하고 싶으면 둘 다 키우고 페널티도 같이 키울 것.
- `low-battery-move-penalty`: 0이면 배터리 페널티 OFF (관측에는 여전히 보임). 0.2 ~ 0.4 정도가 강한 신호.

In [ ]:
!python comm_train_battery.py \
  --grid-size 25 \
  --n-agents 14 \
  --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 \
  --shaping-coef 0.5 \
  --initial-battery 1.0 \
  --hover-battery-cost 0.002 \
  --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --total-frames 1048576 \
  --frames-per-batch 4096 \
  --minibatch-size 256 \
  --ppo-epochs 3 \
  --lr 5e-4 \
  --ent-coef 0.03 \
  --ckpt-every 5 \
  --save-dir checkpoints_battery_ground_A \
  --tb-logdir runs_battery_ground_A

이어 학습용 (ckpt 번호는 실제로 생성된 값으로 바꾸기).

In [ ]:
!python comm_train_battery.py \
  --grid-size 25 \
  --n-agents 14 \
  --max-steps 150 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.2 \
  --hover-penalty 0.03 \
  --shaping-coef 0.3 \
  --initial-battery 1.0 \
  --hover-battery-cost 0.002 \
  --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.2 \
  --load-ckpt checkpoints_battery_ground_A/ckpt_170.pt \
  --total-frames 262144 \
  --frames-per-batch 8192 \
  --minibatch-size 1024 \
  --ppo-epochs 2 \
  --lr 5e-5 \
  --ent-coef 0.003 \
  --clip-eps 0.05 \
  --ckpt-every 5 \
  --save-dir checkpoints_battery_ground_A_resume \
  --tb-logdir runs_battery_ground_A_resume

## 7. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs_battery_ground_A

## 8. 평가 + 배터리 표시 GIF

각 드론 위에 작은 배터리 막대(녹→황→적)와 퍼센트가 표시됨. 결과 텍스트 파일에는 평균/최소/표준편차(낮을수록 균등 사용)가 같이 남음.

`ckpt_NN.pt`는 실제 생성된 체크포인트 번호로 바꿀 것.

In [ ]:
!python comm_eval_battery.py \
  --ckpt checkpoints_battery_fast/ckpt_8.pt \
  --grid-size 25 \
  --n-agents 14 \
  --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --initial-battery 1.0 \
  --hover-battery-cost 0.002 \
  --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.1 \
  --greedy \
  --n-episodes 50 \
  --save-gif demo_battery_ground_A.gif \
  --out eval_battery_ground_A.txt

## 9. 결과를 Google Drive로 복사

In [ ]:
!cp -v *.gif /content/drive/MyDrive/drone_results/battery/gifs/ 2>/dev/null || echo "no gifs"
!cp -v eval_battery_*.txt /content/drive/MyDrive/drone_results/battery/evals/ 2>/dev/null || echo "no eval txts"
!cp -rv checkpoints_battery_* /content/drive/MyDrive/drone_results/battery/ckpts/ 2>/dev/null || echo "no ckpt dirs"

## 10. 다단계 전환 (배터리 인지)

기본 단일 단계가 학습되고 나면 여러 모양으로 확장. 다단계는 에피소드가 길어지므로 배터리가 많이 소모됨.

In [ ]:
!python comm_train_battery.py \
  --grid-size 25 \
  --n-agents 14 \
  --max-steps 200 \
  --shapes GROUND,A,B,C,D \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 \
  --shaping-coef 0.5 \
  --initial-battery 1.0 \
  --hover-battery-cost 0.002 \
  --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --total-frames 1048576 \
  --frames-per-batch 4096 \
  --minibatch-size 256 \
  --ppo-epochs 3 \
  --lr 5e-4 \
  --ent-coef 0.03 \
  --ckpt-every 5 \
  --save-dir checkpoints_battery_ABCD \
  --tb-logdir runs_battery_ABCD

In [ ]:
!python comm_eval_battery.py \
  --ckpt checkpoints_battery_ABCD/ckpt_5.pt \
  --grid-size 25 \
  --n-agents 14 \
  --max-steps 200 \
  --shapes GROUND,A,B,C,D \
  --completion-reward 100.0 \
  --initial-battery 1.0 \
  --hover-battery-cost 0.002 \
  --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --greedy \
  --n-episodes 1 \
  --save-gif demo_battery_ABCD.gif \
  --out eval_battery_ABCD.txt

## 11. (선택) 배터리 OFF 비교 학습

배터리 미고려 vs 고려 비교용. `low-battery-move-penalty 0`이면 페널티는 끄고 관측의 배터리 슬롯만 살아있는 상태로 학습 가능 (관측 차원 동일). 완전히 배터리를 무시하고 비교하려면 기존 `comm_train.py`를 사용하면 됨.

In [ ]:
# 완전 배터리 미고려 baseline (기존 스크립트)
!python comm_train.py \
  --grid-size 25 \
  --n-agents 14 \
  --max-steps 120 \
  --shapes GROUND,A \
  --completion-reward 100.0 \
  --assigned-target-reward 0.1 \
  --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 \
  --shaping-coef 0.5 \
  --total-frames 1048576 \
  --frames-per-batch 4096 \
  --minibatch-size 256 \
  --ppo-epochs 3 \
  --lr 5e-4 \
  --ent-coef 0.03 \
  --ckpt-every 5 \
  --save-dir checkpoints_baseline_ground_A \
  --tb-logdir runs_baseline_ground_A